# Sales training with 30 reps: power after the study

**Sample Size and Power Calculation** · a worked example from the StatNex teaching guide

**The story.** A company sent 30 sales reps on a short training course and compared their weekly sales before and after. The file shows the first and last three reps; the whole study is summarised in the settings: average gain \$0.71k per week, standard deviation of the gains \$1.9k.

**The question.** With 30 reps and gains this noisy, how likely was the study to detect the effect at all?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `13_sample_size/sample_size__power.csv` (7 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Rep,Before training,After training,Sales gain
No.1,12.5,13.2,0.7
No.2,13,16.3,3.3
No.3,13.9,13,-0.9
...,...,...,...
No.28,9.4,8.6,-0.8
No.29,14.7,17.2,2.5
No.30,11.5,14.1,2.6

## Libraries

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import optimize, stats
from statsmodels.stats.power import TTestIndPower, TTestPower

pd.set_option("display.width", 120)

# No data file: everything comes from the numbers set below.
alpha = 5 / 100
sides = ["two-sided", "one-sided"]


def z(p):
    """The standard normal value with probability p above it."""
    return stats.norm.isf(p)

## Power of the study as run

In [ ]:
# The power is the chance that a study of this size would reach significance if the
# true effect were exactly the one observed. Worked out from the study's own result like
# this, it is only a restatement of the P value (P near 0.05 always gives power near
# 0.5); use it to plan the next study, not to explain this one.
n1 = 30


def t_power(effect, two_sided):
    """Power of the t test against Cohen's d = effect, from statsmodels."""
    # statsmodels reads it off the noncentral t distribution: the distribution the t
    # statistic follows when the true difference is the one observed. Two-sided power
    # counts both tails. One-sided, in the direction the data point, as the one-sided P
    # value below is.
    side = "two-sided" if two_sided else "larger"
    size = effect if two_sided else abs(effect)
    if n2 is None:
        return TTestPower().power(effect_size=size, nobs=n1, alpha=alpha, alternative=side)
    return TTestIndPower().power(effect_size=size, nobs1=n1, ratio=n2 / n1, alpha=alpha,
                                 alternative=side)


# The mean and the SD are those of the paired differences.
difference, sd = 0.71, 1.9
error = sd / math.sqrt(n1)
df = n1 - 1
statistic = difference / error
# Cohen's d: the difference in standard deviations. The noncentrality is the observed t.
effect = difference / sd
n2 = None
power_table = pd.DataFrame([[t_power(effect, True), t_power(effect, False)],
                            [statistic, statistic]],
                           index=["power", "noncentrality"], columns=sides)
print(power_table)

## The test itself

In [ ]:
# The confidence interval is always 95%, whatever the significance level.
critical = stats.t.isf(0.025, df)
detail = pd.Series({
    "difference": difference,
    "standard error": error,
    "t": statistic, "df": df,
    "P two-sided": 2 * stats.t.sf(abs(statistic), df), "P one-sided": stats.t.sf(abs(statistic), df),
    "95% CI lower": difference - critical * error, "95% CI upper": difference + critical * error,
})
print()
print(detail)

## Benchmarks for effect sizes (Cohen)

In [ ]:
# Rules of thumb for when nothing better is known: plan from the smallest effect that
# matters to the decision instead. The Cramer's V row repeats Cohen's values for w,
# which hold for V only when the smaller side of the table has two categories; for a
# larger table divide them by sqrt(min(rows, columns) - 1).
benchmarks = pd.DataFrame(
    [["t test", "r", 0.1, 0.3, 0.5], ["t test", "d", 0.2, 0.5, 0.8],
     ["ANOVA", "eta^2", 0.01, 0.06, 0.14], ["correlation", "r", 0.1, 0.3, 0.5],
     ["chi-square (2x2)", "phi", 0.1, 0.3, 0.5], ["chi-square (other)", "Cramer's V", 0.1, 0.3, 0.5],
     ["non-parametric", "r", 0.1, 0.3, 0.5], ["multiple regression", "R^2", 0.02, 0.13, 0.26]],
    columns=["analysis", "index", "small", "medium", "large"],
)
print()
print(benchmarks.to_string(index=False))

## What the result says

Power is the chance that a study detects an effect if the effect is real. With 30 reps, an average gain of 0.71 (\$000 a week) and a standard deviation of the gains of 1.9, the two-sided power is 0.51. Even if the training truly adds this much, such a study would show it only about half the time.

The breakdown of the test shows how close this study was. The standard error, the typical chance error of the average gain, is 0.35, so the t statistic is 2.05. The two-sided P value is 0.0498, just under the 5% significance level. The 95% confidence interval, the range of values the data agree with for the true gain, runs from 0.0005 to 1.42.

So the study barely had a fair chance. It did show a gain, but only just, and the real gain could be almost nothing or quite large.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.